https://clinicaltrials.gov/data-api/api

In [ ]:
from __future__ import annotations

import re
import time
from dataclasses import asdict, dataclass, field
from datetime import date
from typing import Any, Dict, Iterable, List, Literal, Optional

import requests

In [ ]:
# =========================
# Shared normalized schema
# =========================

SourceType = Literal["clinicaltrials_gov", "pubmed"]
RecordType = Literal["trial", "publication"]


@dataclass
class QueryFilters:
    indication: Optional[str] = None
    phase: Optional[List[str]] = None
    geography: Optional[List[str]] = None          # country names
    min_sample_size: Optional[int] = None
    max_sample_size: Optional[int] = None
    date_from: Optional[str] = None                # ISO: YYYY-MM-DD
    date_to: Optional[str] = None                  # ISO: YYYY-MM-DD
    status: Optional[List[str]] = None             # e.g. ["RECRUITING"]
    sex: Optional[str] = None                      # PubMed-friendly: "male" / "female"
    age_group: Optional[str] = None                # PubMed-friendly: "adult" / "child" / etc.
    study_terms: Optional[List[str]] = None        # extra free-text terms


@dataclass
class NormalizedRecord:
    source: SourceType
    record_type: RecordType

    # source identifiers
    source_id: str
    secondary_ids: List[str] = field(default_factory=list)

    # titles
    title: Optional[str] = None
    official_title: Optional[str] = None

    # medical/topic fields
    indication: List[str] = field(default_factory=list)
    intervention_names: List[str] = field(default_factory=list)
    phase: List[str] = field(default_factory=list)

    # operational fields
    status: Optional[str] = None
    geography: List[str] = field(default_factory=list)
    sample_size: Optional[int] = None

    # time fields
    start_date: Optional[str] = None
    completion_date: Optional[str] = None
    primary_completion_date: Optional[str] = None
    publication_date: Optional[str] = None

    # publication / linking
    url: Optional[str] = None
    abstract: Optional[str] = None
    linked_trial_ids: List[str] = field(default_factory=list)

    # provenance / quality
    source_confidence: Literal["high", "medium", "low"] = "high"
    missing_fields: List[str] = field(default_factory=list)
    raw: Dict[str, Any] = field(default_factory=dict)






In [ ]:
# =========================
# Utilities
# =========================

def _compact_none_fields(record: NormalizedRecord) -> NormalizedRecord:
    missing = []
    for field_name in [
        "title", "official_title", "status", "sample_size",
        "start_date", "completion_date", "primary_completion_date",
        "publication_date", "abstract"
    ]:
        if getattr(record, field_name) in (None, [], ""):
            missing.append(field_name)

    if not record.phase:
        missing.append("phase")
    if not record.geography:
        missing.append("geography")
    if not record.indication:
        missing.append("indication")

    record.missing_fields = sorted(set(missing))
    return record


def _safe_get(dct: Dict[str, Any], *path: str, default=None):
    cur = dct
    for key in path:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key)
        if cur is None:
            return default
    return cur


def _normalize_date_str(value: Optional[str]) -> Optional[str]:
    if not value:
        return None
    # ClinicalTrials.gov already uses ISO-like strings for modern API fields.
    # PubMed dates often arrive differently; keep raw string if exact parsing is not safe.
    return value.strip()


def _extract_nct_ids(text: Optional[str]) -> List[str]:
    if not text:
        return []
    return sorted(set(re.findall(r"\bNCT\d{8}\b", text, flags=re.IGNORECASE)))



In [ ]:
# =========================
# ClinicalTrials.gov client
# =========================

class ClinicalTrialsGovClient:
    BASE_URL = "https://clinicaltrials.gov/api/v2"

    def __init__(self, session: Optional[requests.Session] = None, timeout: int = 30):
        self.session = session or requests.Session()
        self.timeout = timeout

    def search_trials(
        self,
        filters: QueryFilters,
        page_size: int = 100,
        max_pages: int = 5,
    ) -> List[NormalizedRecord]:
        """
        Query ClinicalTrials.gov /studies and normalize records.

        Notes:
        - This API is the authoritative source for structured trial fields.
        - Geography support here is reliable at location-country level.
        - Sample size maps from enrollment count when present.
        """
        params: Dict[str, Any] = {
            "format": "json",
            "countTotal": "true",
            "pageSize": min(page_size, 1000),
        }

        # Query / filter construction
        if filters.indication:
            params["query.cond"] = filters.indication

        if filters.status:
            params["filter.overallStatus"] = ",".join(filters.status)

        # There is no simple country-list filter parameter in the official core docs snippet.
        # A practical approach is to search broadly and post-filter on location countries.
        #
        # If you later want geospatial radius filtering, the API supports filter.geo distance syntax.
        # That is useful for coordinates, not for country lists.

        # Restrict returned payload to the fields we actually need.
        params["fields"] = ",".join([
            "protocolSection.identificationModule.nctId",
            "protocolSection.identificationModule.briefTitle",
            "protocolSection.identificationModule.officialTitle",
            "protocolSection.conditionsModule.conditions",
            "protocolSection.armsInterventionsModule.interventions",
            "protocolSection.designModule.phases",
            "protocolSection.statusModule.overallStatus",
            "protocolSection.statusModule.startDateStruct",
            "protocolSection.statusModule.completionDateStruct",
            "protocolSection.statusModule.primaryCompletionDateStruct",
            "protocolSection.contactsLocationsModule.locations",
            "protocolSection.designModule.enrollmentInfo",
        ])

        out: List[NormalizedRecord] = []
        page_token: Optional[str] = None
        pages = 0

        while pages < max_pages:
            if page_token:
                params["pageToken"] = page_token

            response = self.session.get(
                f"{self.BASE_URL}/studies",
                params=params,
                timeout=self.timeout,
            )
            response.raise_for_status()
            payload = response.json()

            studies = payload.get("studies", [])
            for study in studies:
                rec = self._normalize_study(study)

                if filters.phase:
                    wanted = {p.lower() for p in filters.phase}
                    actual = {p.lower() for p in rec.phase}
                    if not actual.intersection(wanted):
                        continue

                if filters.geography:
                    wanted_countries = {c.lower() for c in filters.geography}
                    actual_countries = {c.lower() for c in rec.geography}
                    if not actual_countries.intersection(wanted_countries):
                        continue

                if filters.min_sample_size is not None:
                    if rec.sample_size is None or rec.sample_size < filters.min_sample_size:
                        continue

                if filters.max_sample_size is not None:
                    if rec.sample_size is None or rec.sample_size > filters.max_sample_size:
                        continue

                out.append(rec)

            page_token = payload.get("nextPageToken")
            if not page_token:
                break

            pages += 1

        return out

    def _normalize_study(self, study: Dict[str, Any]) -> NormalizedRecord:
        identification = _safe_get(study, "protocolSection", "identificationModule", default={}) or {}
        status_mod = _safe_get(study, "protocolSection", "statusModule", default={}) or {}
        design_mod = _safe_get(study, "protocolSection", "designModule", default={}) or {}
        conditions_mod = _safe_get(study, "protocolSection", "conditionsModule", default={}) or {}
        contacts_mod = _safe_get(study, "protocolSection", "contactsLocationsModule", default={}) or {}
        arms_mod = _safe_get(study, "protocolSection", "armsInterventionsModule", default={}) or {}

        nct_id = identification.get("nctId")
        brief_title = identification.get("briefTitle")
        official_title = identification.get("officialTitle")

        conditions = conditions_mod.get("conditions") or []

        interventions = []
        for item in arms_mod.get("interventions") or []:
            name = item.get("name")
            if name:
                interventions.append(name)

        phases = design_mod.get("phases") or []

        locations = contacts_mod.get("locations") or []
        countries = sorted({
            loc.get("country")
            for loc in locations
            if isinstance(loc, dict) and loc.get("country")
        })

        enrollment_info = design_mod.get("enrollmentInfo") or {}
        sample_size = enrollment_info.get("count")
        try:
            sample_size = int(sample_size) if sample_size is not None else None
        except (ValueError, TypeError):
            sample_size = None

        rec = NormalizedRecord(
            source="clinicaltrials_gov",
            record_type="trial",
            source_id=nct_id or "",
            secondary_ids=[],
            title=brief_title,
            official_title=official_title,
            indication=conditions,
            intervention_names=interventions,
            phase=phases,
            status=status_mod.get("overallStatus"),
            geography=countries,
            sample_size=sample_size,
            start_date=_normalize_date_str(_safe_get(status_mod, "startDateStruct", "date")),
            completion_date=_normalize_date_str(_safe_get(status_mod, "completionDateStruct", "date")),
            primary_completion_date=_normalize_date_str(_safe_get(status_mod, "primaryCompletionDateStruct", "date")),
            publication_date=None,
            url=f"https://clinicaltrials.gov/study/{nct_id}" if nct_id else None,
            abstract=None,
            linked_trial_ids=[nct_id] if nct_id else [],
            source_confidence="high",
            raw=study,
        )
        return _compact_none_fields(rec)


In [ ]:
# =========================
# PubMed client
# =========================

class PubMedClient:
    EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

    def __init__(
        self,
        email: str,
        tool: str = "trial_aggregator",
        api_key: Optional[str] = None,
        session: Optional[requests.Session] = None,
        timeout: int = 30,
    ):
        self.email = email
        self.tool = tool
        self.api_key = api_key
        self.session = session or requests.Session()
        self.timeout = timeout

    def search_publications(
        self,
        filters: QueryFilters,
        retmax: int = 100,
    ) -> List[NormalizedRecord]:
        """
        Query PubMed using ESearch + EFetch and normalize records.

        Important:
        PubMed is publication-centric, not a structured trial registry.
        Phase, sample size, and geography are often unavailable or only inferable.
        """
        term = self._build_pubmed_query(filters)

        id_payload = self._esearch(term=term, retmax=retmax)
        ids = id_payload.get("esearchresult", {}).get("idlist", [])
        if not ids:
            return []

        articles = self._efetch_pubmed_xml(ids)
        records = [self._normalize_pubmed_article(article) for article in articles]

        # Optional post-filter for geography against affiliation/abstract text if user insists.
        # This is weaker than ClinicalTrials.gov country filtering.
        if filters.geography:
            wanted = {g.lower() for g in filters.geography}
            filtered = []
            for rec in records:
                hay = " ".join([
                    rec.title or "",
                    rec.abstract or "",
                    " ".join(rec.geography),
                ]).lower()
                if any(country.lower() in hay for country in wanted):
                    filtered.append(rec)
            records = filtered

        return records

    def _common_params(self) -> Dict[str, str]:
        params = {
            "tool": self.tool,
            "email": self.email,
        }
        if self.api_key:
            params["api_key"] = self.api_key
        return params

    def _build_pubmed_query(self, filters: QueryFilters) -> str:
        parts: List[str] = []

        # indication/topic
        if filters.indication:
            parts.append(f'("{filters.indication}"[Title/Abstract] OR "{filters.indication}"[MeSH Terms])')

        if filters.study_terms:
            for term in filters.study_terms:
                parts.append(f'("{term}"[Title/Abstract])')

        # phase approximation
        # PubMed does not expose phase as a first-class structured field in the way CT.gov does.
        if filters.phase:
            phase_clauses = []
            for phase in filters.phase:
                phase_norm = phase.lower().replace(" ", "")
                if "1/2" in phase_norm or "phase1/phase2" in phase_norm:
                    phase_clauses.append('"phase i/ii"[Title/Abstract]')
                    phase_clauses.append('"phase 1/2"[Title/Abstract]')
                elif "2/3" in phase_norm:
                    phase_clauses.append('"phase ii/iii"[Title/Abstract]')
                    phase_clauses.append('"phase 2/3"[Title/Abstract]')
                elif "1" in phase_norm:
                    phase_clauses.append('"phase i"[Title/Abstract]')
                    phase_clauses.append('"phase 1"[Title/Abstract]')
                elif "2" in phase_norm:
                    phase_clauses.append('"phase ii"[Title/Abstract]')
                    phase_clauses.append('"phase 2"[Title/Abstract]')
                elif "3" in phase_norm:
                    phase_clauses.append('"phase iii"[Title/Abstract]')
                    phase_clauses.append('"phase 3"[Title/Abstract]')
                elif "4" in phase_norm:
                    phase_clauses.append('"phase iv"[Title/Abstract]')
                    phase_clauses.append('"phase 4"[Title/Abstract]')
            if phase_clauses:
                parts.append("(" + " OR ".join(sorted(set(phase_clauses))) + ")")

        # publication types related to trials
        parts.append('(clinical trial[Publication Type] OR randomized controlled trial[Publication Type] OR trial[Title])')

        # date filter
        if filters.date_from or filters.date_to:
            lo = filters.date_from or "1000/01/01"
            hi = filters.date_to or date.today().strftime("%Y/%m/%d")
            parts.append(f'("{lo}"[Date - Publication] : "{hi}"[Date - Publication])')

        # sex and age are PubMed-native filters, but indexing-dependent
        if filters.sex:
            sex_map = {
                "male": "male[MeSH Terms]",
                "female": "female[MeSH Terms]",
            }
            clause = sex_map.get(filters.sex.lower())
            if clause:
                parts.append(clause)

        if filters.age_group:
            age_map = {
                "child": "child[MeSH Terms]",
                "adolescent": "adolescent[MeSH Terms]",
                "adult": "adult[MeSH Terms]",
                "aged": "aged[MeSH Terms]",
            }
            clause = age_map.get(filters.age_group.lower())
            if clause:
                parts.append(clause)

        return " AND ".join(parts) if parts else "clinical trial[Publication Type]"

    def _esearch(self, term: str, retmax: int) -> Dict[str, Any]:
        params = {
            **self._common_params(),
            "db": "pubmed",
            "term": term,
            "retmode": "json",
            "retmax": str(retmax),
            "sort": "pub date",
        }
        resp = self.session.get(
            f"{self.EUTILS_BASE}/esearch.fcgi",
            params=params,
            timeout=self.timeout,
        )
        resp.raise_for_status()
        return resp.json()

    def _efetch_pubmed_xml(self, ids: List[str]) -> List[Dict[str, Any]]:
        """
        Minimal XML retrieval with light parsing via ElementTree.
        """
        import xml.etree.ElementTree as ET

        params = {
            **self._common_params(),
            "db": "pubmed",
            "id": ",".join(ids),
            "retmode": "xml",
        }
        resp = self.session.get(
            f"{self.EUTILS_BASE}/efetch.fcgi",
            params=params,
            timeout=self.timeout,
        )
        resp.raise_for_status()

        root = ET.fromstring(resp.content)
        articles: List[Dict[str, Any]] = []

        for pubmed_article in root.findall(".//PubmedArticle"):
            medline = pubmed_article.find("MedlineCitation")
            article = medline.find("Article") if medline is not None else None
            pubmed_data = pubmed_article.find("PubmedData")

            pmid = medline.findtext("PMID") if medline is not None else None
            title = article.findtext("ArticleTitle") if article is not None else None

            abstract_parts = []
            if article is not None:
                for elem in article.findall("./Abstract/AbstractText"):
                    text = "".join(elem.itertext()).strip()
                    if text:
                        abstract_parts.append(text)
            abstract = "\n".join(abstract_parts) if abstract_parts else None

            journal_issue = article.find("./Journal/JournalIssue/PubDate") if article is not None else None
            pub_date = None
            if journal_issue is not None:
                year = journal_issue.findtext("Year")
                month = journal_issue.findtext("Month")
                day = journal_issue.findtext("Day")
                pub_date = " ".join(x for x in [year, month, day] if x)

            mesh_terms = []
            if medline is not None:
                for mh in medline.findall("./MeshHeadingList/MeshHeading/DescriptorName"):
                    txt = "".join(mh.itertext()).strip()
                    if txt:
                        mesh_terms.append(txt)

            affiliations = []
            if article is not None:
                for aff in article.findall(".//AffiliationInfo/Affiliation"):
                    txt = "".join(aff.itertext()).strip()
                    if txt:
                        affiliations.append(txt)

            article_ids = []
            if pubmed_data is not None:
                for aid in pubmed_data.findall(".//ArticleId"):
                    id_type = aid.attrib.get("IdType")
                    value = "".join(aid.itertext()).strip()
                    if value:
                        article_ids.append({"type": id_type, "value": value})

            articles.append({
                "pmid": pmid,
                "title": title,
                "abstract": abstract,
                "publication_date": pub_date,
                "mesh_terms": mesh_terms,
                "affiliations": affiliations,
                "article_ids": article_ids,
            })

        return articles

    def _normalize_pubmed_article(self, article: Dict[str, Any]) -> NormalizedRecord:
        abstract = article.get("abstract")
        title = article.get("title")
        pmid = article.get("pmid")

        combined_text = " ".join(x for x in [title, abstract] if x)
        linked_trial_ids = _extract_nct_ids(combined_text)

        # Geography here is weak and inferred only from affiliations.
        affiliations = article.get("affiliations") or []
        geography = affiliations[:]

        secondary_ids = []
        for aid in article.get("article_ids", []):
            secondary_ids.append(f"{aid.get('type')}:{aid.get('value')}")

        rec = NormalizedRecord(
            source="pubmed",
            record_type="publication",
            source_id=str(pmid),
            secondary_ids=secondary_ids,
            title=title,
            official_title=None,
            indication=article.get("mesh_terms") or [],
            intervention_names=[],
            phase=[],
            status=None,
            geography=geography,
            sample_size=None,
            start_date=None,
            completion_date=None,
            primary_completion_date=None,
            publication_date=_normalize_date_str(article.get("publication_date")),
            url=f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else None,
            abstract=abstract,
            linked_trial_ids=linked_trial_ids,
            source_confidence="medium" if linked_trial_ids else "low",
            raw=article,
        )
        return _compact_none_fields(rec)

In [ ]:
# =========================
# Aggregation layer
# =========================

def aggregate_records(*record_lists: Iterable[NormalizedRecord]) -> List[Dict[str, Any]]:
    """
    Flatten multiple normalized streams into one list of plain dicts.
    Deduping strategy can be improved later. Current logic:
    - trials dedupe by NCT ID
    - publications dedupe by PMID
    """
    seen = set()
    output = []

    for records in record_lists:
        for record in records:
            key = (record.source, record.source_id)
            if key in seen:
                continue
            seen.add(key)
            output.append(asdict(record))

    return output

In [ ]:
# =========================
# Example usage
# =========================

if __name__ == "__main__":
    filters = QueryFilters(
        indication="non-small cell lung cancer",
        phase=["PHASE3"],
        geography=["Germany", "United States"],
        min_sample_size=100,
        date_from="2022-01-01",
        date_to="2026-04-09",
        study_terms=["PD-1", "immunotherapy"],
    )

    ctg = ClinicalTrialsGovClient()
    pubmed = PubMedClient(
        email="your_email@example.com",
        tool="trial_aggregator",
        api_key=None,  # optional but recommended if higher throughput is needed
    )

    ctg_records = ctg.search_trials(filters=filters, page_size=100, max_pages=3)
    pubmed_records = pubmed.search_publications(filters=filters, retmax=100)

    combined = aggregate_records(ctg_records, pubmed_records)

    print(f"ClinicalTrials.gov records: {len(ctg_records)}")
    print(f"PubMed records: {len(pubmed_records)}")
    print(f"Combined records: {len(combined)}")

    # Example: print first 2 normalized rows
    for row in combined[:2]:
        print(row)